# OLS IMPLEMENTATION

## Libraries

In [85]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from LinearRegression import OLS

## Setup

In [86]:
train_test_split_percent = 0.8
y_col_name = "Price"

In [87]:
def fill_null_num_col(frame: pd.DataFrame, val: float | int, cols: [String]):    
    for col_name in cols:
        if col_name not in num_cols:
            continue
            
        frame.fillna({col_name: val}, inplace=True)
    return frame

In [88]:
def fill_null_num_col_median(frame: pd.DataFrame, cols: [String]):    
    for col_name in cols:
        if col_name not in num_cols:
            continue
            
        frame.fillna({col_name: frame[col_name].median()}, inplace=True)
    pass

In [89]:
def fill_null_cat_col(frame: pd.DataFrame, val: String, cols: [String]):
    for col_name in cols:
        if col_name not in cat_cols:
            continue

        frame.fillna({col_name: val}, inplace=True)

In [90]:
def one_hot_encoding(frame: pd.DataFrame, columns: [String] = [], drop_first: bool = False):
    return pd.get_dummies(frame, columns=columns, drop_first=drop_first, dtype=int)

In [91]:
def split_X_y(frame:pd.DataFrame, y: String):
    frame_cols = frame.columns.to_list()
    if y not in frame_cols:
        print(f"Column y: {y} can't be found in dataframe.")
        return None
    return frame.pop(y)

## Data exploration

In [92]:
df = pd.read_csv("./vietnam_housing_dataset.csv")
df_shape = df.shape
df_height = df_shape[0]
df_weidth = df_shape[1]

cols = df.columns.to_list()
num_cols = df.select_dtypes(include="number").columns.to_list()
cat_cols = df.select_dtypes(include=["category", "object", "str"]).columns.to_list()

print(df_shape)
print(f"Numeric columns:\n{num_cols}")
print("\n")
print(f"Categorical columns:\n{cat_cols}")

(30229, 12)
Numeric columns:
['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']


Categorical columns:
['Address', 'House direction', 'Balcony direction', 'Legal status', 'Furniture state']


In [93]:
print(df.head())

                                             Address  Area  Frontage  \
0  Dự án The Empire - Vinhomes Ocean Park 2, Xã L...  84.0       NaN   
1  Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...  60.0       NaN   
2  Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...  90.0       6.0   
3  Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...  54.0       NaN   
4   Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh  92.0       NaN   

   Access Road House direction Balcony direction  Floors  Bedrooms  Bathrooms  \
0          NaN             NaN               NaN     4.0       NaN        NaN   
1          NaN             NaN               NaN     5.0       NaN        NaN   
2         13.0      Đông - Bắc        Đông - Bắc     5.0       NaN        NaN   
3          3.5       Tây - Nam         Tây - Nam     2.0       2.0        3.0   
4          NaN      Đông - Nam        Đông - Nam     2.0       4.0        4.0   

       Legal status Furniture state  Price  
0  Have certificate             NaN

In [94]:
print(f"Null (%) for each column:\n{(df.isnull().mean()*100).round(2)}")

Null (%) for each column:
Address               0.00
Area                  0.00
Frontage             38.25
Access Road          43.99
House direction      70.26
Balcony direction    82.65
Floors               11.92
Bedrooms             17.08
Bathrooms            23.40
Legal status         14.91
Furniture state      46.71
Price                 0.00
dtype: float64


## Data processing

### Preparations

In [95]:
train_test_split_amount = int(df_height*train_test_split_percent)

# Skull
df['District'] = df['Address'].astype(str).apply(
    lambda x: x.split(',')[-2].strip() if len(x.split(',')) >= 2 else 'Other'
)
df = df.drop(columns=["Address"])
display(df)

,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price,District
0,84.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60,Văn Giang
1,60.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50,Văn Giang
2,90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90,Văn Giang
3,54.0,NaN,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35,Gò Vấp
4,92.0,NaN,NaN,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90,Gò Vấp
...,...,...,...,...,...,...,...,...,...,...,...,...
30224,67.0,4.1,16.0,NaN,NaN,1.0,3.0,2.0,Have certificate,NaN,4.60,Gò Vấp
30225,30.0,NaN,NaN,NaN,NaN,5.0,3.0,3.0,Have certificate,NaN,4.70,Long Biên
30226,69.4,4.0,15.0,Đông - Bắc,Đông - Bắc,NaN,NaN,NaN,Have certificate,Basic,7.50,Thủ Đức
30227,96.0,NaN,8.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,9.50,Gò Vấp


In [96]:
# cat_cols.append("District")
# cat_cols.remove("Address")
cat_cols = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state', 'District']#df.select_dtypes(include=["category", "object", "str"]).columns.to_list()

In [97]:
null_num_cols = df.select_dtypes(include="number").isnull().columns.tolist()
null_cat_cols = df.select_dtypes(include="category").isnull().columns.tolist()

print(f"Numeric columns with null:\n{null_num_cols}")
print("\n")
print(f"Categorical columns with null:\n{null_cat_cols}")

Numeric columns with null:
['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']


Categorical columns with null:
[]


### Categorical features

In [98]:
# Since categorical must be in all, we process cats before splitting
fill_null_cat_col(df, "Unknown", cat_cols)
df_encoded = one_hot_encoding(df, cat_cols, True)

In [99]:
train = df_encoded.iloc[:train_test_split_amount].copy()
test = df_encoded.iloc[train_test_split_amount:].copy()

print(train.shape[0])
print(test.shape[0])
print(f"{df_height}:{train.shape[0]+test.shape[0]}")

24183
6046
30229:30229


Numeric columns:
['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']

Categorical columns:
['District', 'House direction', 'Balcony direction', 'Legal status', 'Furniture state']

### Numerical features

In [100]:
num_col_fill_median = ["Area", "Frontage", "Access Road", "Price"]
num_col_fill_one = ["Floors", "Bedrooms", "Bathrooms"]
print(f"{len(num_col_fill_median+num_col_fill_one)}:{len(num_cols)}")

7:7


In [101]:
# Train
fill_null_num_col_median(train, num_col_fill_median)
fill_null_num_col(train, 1, num_col_fill_one)

# Test
fill_null_num_col_median(test, num_col_fill_median)
fill_null_num_col(test, 1, num_col_fill_one)

,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Price,House direction_Nam,House direction_Tây,House direction_Tây - Bắc,...,District_Đơn Dương,District_Đảo Phú Quý,District_Đất Đỏ,District_Định Quán,District_Đống Đa,District_Đồ Sơn,District_Đồng Xoài,District_Đức Hòa,District_Đức Trọng,District_Ứng Hòa
24183,37.0,4.5,5.5,1.0,1.0,1.0,6.55,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24184,29.0,3.7,2.5,4.0,3.0,3.0,2.55,0,0,1,...,0,0,0,0,0,0,0,0,0,0
24185,117.0,6.0,16.5,1.0,1.0,1.0,7.40,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24186,75.0,4.5,5.5,4.0,1.0,1.0,4.25,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24187,75.0,5.0,5.5,4.0,4.0,5.0,6.35,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30224,67.0,4.1,16.0,1.0,3.0,2.0,4.60,0,0,0,...,0,0,0,0,0,0,0,0,0,0
30225,30.0,4.5,5.5,5.0,3.0,3.0,4.70,0,0,0,...,0,0,0,0,0,0,0,0,0,0
30226,69.4,4.0,15.0,1.0,1.0,1.0,7.50,0,0,0,...,0,0,0,0,0,0,0,0,0,0
30227,96.0,4.5,8.0,4.0,1.0,1.0,9.50,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Split

In [102]:
y_train = split_X_y(train, y_col_name)
y_test = split_X_y(test, y_col_name)

In [103]:
X_train = train
X_test = test

display(X_train)

,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,House direction_Nam,House direction_Tây,House direction_Tây - Bắc,House direction_Tây - Nam,...,District_Đơn Dương,District_Đảo Phú Quý,District_Đất Đỏ,District_Định Quán,District_Đống Đa,District_Đồ Sơn,District_Đồng Xoài,District_Đức Hòa,District_Đức Trọng,District_Ứng Hòa
0,84.0,4.5,6.0,4.0,1.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,60.0,4.5,6.0,5.0,1.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,90.0,6.0,13.0,5.0,1.0,1.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,54.0,4.5,3.5,2.0,2.0,3.0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,92.0,4.5,6.0,2.0,4.0,4.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24178,60.0,4.0,6.0,2.0,2.0,2.0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
24179,52.8,8.0,8.0,3.0,3.0,3.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24180,40.2,4.5,6.0,7.0,5.0,6.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24181,40.0,3.5,8.0,4.0,4.0,3.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [104]:
print(y_train)
print(y_test)

0        8.60
1        7.50
2        8.90
3        5.35
4        6.90
         ... 
24178    3.90
24179    4.50
24180    8.90
24181    5.40
24182    2.80
Name: Price, Length: 24183, dtype: float64
24183    6.55
24184    2.55
24185    7.40
24186    4.25
24187    6.35
         ... 
30224    4.60
30225    4.70
30226    7.50
30227    9.50
30228    3.15
Name: Price, Length: 6046, dtype: float64


## Training

In [105]:
model = OLS.OLS()

In [106]:
model.fit(X_train, y_train)

In [107]:
print(model.r2score)

0.4529359810743956


## Evaluation

In [108]:
y_pred = np.asarray(model.predict(X_test)).flatten()

In [109]:
y_true = np.asarray(y_test).flatten()

In [110]:
mae = np.mean(np.abs(y_true-y_pred))
print(f"Mean Absolute Error: {mae}")

Mean Absolute Error: 1.307307396570631


In [111]:
mse = np.mean(np.abs((y_true - y_pred) ** 2))
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 2.7966165227415924
